In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
! pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 845.0 kB/s eta 0:00:000:00:01


In [4]:
from huggingface_hub import notebook_login

notebook_login()

In [5]:
from datasets import load_dataset
imdb = load_dataset('stanfordnlp/imdb')

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
print(imdb['train'][3])
print(imdb)
print(imdb['test'][3])


{'text': "This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been replaced with stupidity. Without going too far on this subject, I would say that follows from the difference in ideals between the French and the Swedish society.<br /><br />A movie of its time, and place. 2/10.", 'label': 0}
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_r

In [7]:
imdb["test"][19422]

{'text': '<br /><br />When this film was released I dismissed as being lightweight pop nonsense. That was a mistake. <br /><br />After repeated viewings and seeing a documentary of the making of DIRTY DANCING, discovering the depth of this film certainly increases its appeal.<br /><br />DIRTY DANCING is a film about change. The evolving nature of relationships within the family, the changes in one\'s view of the world during their coming of age, etc. The story takes place during August of 1963, the final weeks of the last summer of innocence for the American people. The many personal changes experienced by the characters reflect the many changes in American society that would be marked by the Kennedy assassinations and Vietnam.<br /><br />Female movie go\'ers adored this film and repeated trips to the movie houses made it the world\'s most successful dance movie. As a male I find the romantic pairing of ultimate stud Patrick Swayze with very plain Jennifer Grey very hard to accept. Thi

In [8]:
print(imdb['train'].features)

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


"This movie was amazing" <br>
          ↓<br>
      Tokenizer<br>
          ↓<br>
["this", "movie", "was", "amazing"]<br>
          ↓<br>
[2023, 3185, 2001, 6429]<br>

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## testing tokenizer

In [10]:
## here we separate the tokens. 
text = 'this movie is amazing '
token =tokenizer.tokenize(text)
print(token)

['this', 'movie', 'is', 'amazing']


In [11]:
## now we give every token there token_id

token_id = tokenizer.convert_tokens_to_ids(token)
token_id

[2023, 3185, 2003, 6429]

In [12]:
print(tokenizer(text))
# here 101 is CLS token which tell model that this is classification task 
# and 102 is SEP that tell that our sentance is end here 


{'input_ids': [101, 2023, 3185, 2003, 6429, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [13]:
# to call it derctly it does interlly both task token and token_id

tokenizer('hi i am mohit')

{'input_ids': [101, 7632, 1045, 2572, 9587, 16584, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [14]:
def preprocess_function(examples):
    return tokenizer(examples['text'],truncation=True)

In [15]:
imdb["train"][0]["text"]

'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, ev

In [16]:
tokenized_imdb = imdb.map(preprocess_function, batched=True)
print(len(tokenized_imdb['train'][9]['token_type_ids']))
print(len(tokenized_imdb['train'][2]['token_type_ids']))
# now we need to add padding 
print(tokenized_imdb['train'])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

297
133
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})


In [17]:
samples = [
    {
        "input_ids": tokenized_imdb["train"][0]["input_ids"],
        "attention_mask": tokenized_imdb["train"][0]["attention_mask"],
    },
    {
        "input_ids": tokenized_imdb["train"][4]["input_ids"],
        "attention_mask": tokenized_imdb["train"][4]["attention_mask"],
    },
    {
        "input_ids": tokenized_imdb["train"][2]["input_ids"],
        "attention_mask": tokenized_imdb["train"][2]["attention_mask"],
    }
]

for sample in samples:
    print(len(sample["input_ids"]))


363
495
133


In [18]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [19]:
batch = data_collator(samples)
#print(batch)
(batch['input_ids'].shape)

torch.Size([3, 495])

In [20]:
import evaluate
acc = evaluate.load("accuracy")

In [21]:
import numpy as np

def compute_metrics(eval_pred):
    prediction , labels = eval_pred
    prediction = np.argmax(prediction ,axis=1)
    return acc.compute(predictions=prediction,references=labels)
    

In [22]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [23]:
 from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir= "my_first_fine-tune_model",
    learning_rate =2e-5,
    per_device_train_batch_size = 16, #batch size 
    per_device_eval_batch_size= 16,
    num_train_epochs = 2,
    weight_decay  =  0.01, #regularization
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
    
)

In [25]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer, # all vocb token_id 
    data_collator=data_collator, # padding add 
    compute_metrics=compute_metrics # acc calculate function 
)

In [26]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.576299,0.402328,0.920720
2,0.289137,0.399904,0.929040


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1564, training_loss=0.41221144620110006, metrics={'train_runtime': 1787.0963, 'train_samples_per_second': 27.978, 'train_steps_per_second': 0.875, 'total_flos': 6620290065781248.0, 'train_loss': 0.41221144620110006, 'epoch': 2.0})

In [27]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Mohit028/my_first_fine-tune_model/commit/3c5ad3a14aafe7e98fd1a3fa229c8187667f78df', commit_message='End of training', commit_description='', oid='3c5ad3a14aafe7e98fd1a3fa229c8187667f78df', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Mohit028/my_first_fine-tune_model', endpoint='https://huggingface.co', repo_type='model', repo_id='Mohit028/my_first_fine-tune_model'), pr_revision=None, pr_num=None)

In [42]:
text = "This movie is not good but its good but overall experience is not good"

In [30]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="Mohit028/my_first_fine-tune_model")
classifier(text)

config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9954957962036133}]

In [44]:
classifier(text)

[{'label': 'NEGATIVE', 'score': 0.7451605796813965}]